# Scree Plot — Decisão do Número de Componentes SVD

Explora a variância explicada pelo SVD truncado para justificar a escolha
do número de componentes usados na redução dimensional da matriz k-mer.

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from sklearn.decomposition import TruncatedSVD

sns.set_theme(style='whitegrid')
PROC = '../data/processed'
print('Pronto.')

## 1. Carregamento da matriz k-mer

In [ ]:
X = sparse.load_npz(os.path.join(PROC, 'kmer_matrix.npz'))
y = np.load(os.path.join(PROC, 'labels.npy'))
print(f'Matriz k-mer: {X.shape}  (sequências × k-mers)')
print(f'Esparsidade: {100*(1 - X.nnz/(X.shape[0]*X.shape[1])):.2f}%')

## 2. SVD para múltiplos n_components

In [ ]:
max_n = min(600, X.shape[0] - 1, X.shape[1] - 1)
components_range = list(range(10, max_n + 1, 10))

variances = []
print(f'Testando n_components de 10 a {max_n} (passo 10)...')
for n in components_range:
    svd = TruncatedSVD(n_components=n, random_state=42)
    svd.fit(X)
    variances.append(float(svd.explained_variance_ratio_.sum()))

print('Concluído.')

## 3. Scree plot com linha de corte em 95%

In [ ]:
THRESHOLD = 0.95
chosen = None
for n, v in zip(components_range, variances):
    if v >= THRESHOLD:
        chosen = n
        break
if chosen is None:
    chosen = components_range[-1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(components_range, variances, linewidth=2, color='#2c7bb6', marker=None)
ax.axhline(THRESHOLD, linestyle='--', color='#d7191c', linewidth=1.5,
           label=f'Threshold {THRESHOLD:.0%}')
chosen_var = variances[components_range.index(chosen)]
ax.scatter([chosen], [chosen_var], s=150, zorder=5, color='#1a9641',
           label=f'Escolhido: {chosen} componentes ({chosen_var:.2%})')
ax.set_xlabel('Número de componentes SVD', fontsize=12)
ax.set_ylabel('Variância explicada acumulada', fontsize=12)
ax.set_title('Scree Plot — Variância Explicada pelo SVD Truncado', fontsize=14)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/scree_plot_notebook.png', dpi=150)
plt.show()
print(f'Componentes escolhidos: {chosen}  →  variância: {chosen_var:.4f}')

## 4. Tabela: n_components × variância × tamanho em MB

In [ ]:
n_seq = X.shape[0]
bytes_per_float = 4  # float32

rows = []
checkpoints = [10, 25, 50, 100, 150, 200, 300, 400, 500]
for n in checkpoints:
    if n in components_range:
        v = variances[components_range.index(n)]
        mb = n_seq * n * bytes_per_float / 1e6
        rows.append({'n_components': n, 'variância_acumulada': f'{v:.4f}',
                     'variância_%': f'{v*100:.1f}%', 'tamanho_MB': f'{mb:.2f}'})

df_table = pd.DataFrame(rows)
df_table.style.highlight_between(subset=['n_components'],
                                  left=chosen, right=chosen, color='#c8e6c9')

In [ ]:
print(df_table.to_string(index=False))

## 5. Justificativa da escolha de n_components

O critério automático é o **menor número de componentes que supera 95% de
variância explicada acumulada**. Este ponto de corte é amplamente adotado
na literatura de redução dimensional e representa o equilíbrio entre
compressão (eficiência computacional) e preservação de informação estrutural
das sequências proteicas.

A projeção ortogonal via SVD é análoga a transformar um objeto 3D em 2D:
perdemos alguma informação, mas preservamos as relações essenciais. Com 95%
de variância, o sinal discriminativo para identificar biomarcadores é
preservado com alta fidelidade.

## 6. Scatter 2D — Dois primeiros componentes SVD

In [ ]:
svd2 = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd2.fit_transform(X)

fig, ax = plt.subplots(figsize=(8, 6))
for cls, color, label in [(1, '#2c7bb6', 'Positivo'), (0, '#d7191c', 'Negativo')]:
    mask = y == cls
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, label=label,
               alpha=0.5, s=25, edgecolors='none')

ax.set_xlabel(f'SVD-1 ({svd2.explained_variance_ratio_[0]:.2%} variância)', fontsize=11)
ax.set_ylabel(f'SVD-2 ({svd2.explained_variance_ratio_[1]:.2%} variância)', fontsize=11)
ax.set_title('Projeção SVD 2D — Espaço K-mer', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/svd_2d_scatter.png', dpi=150)
plt.show()